In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import datetime

In [0]:
#storage_details
storage_account= "storageretaillakehouse"

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
   "YOUR_STORAGE_KEY"
)


In [0]:
#Parameters
dbutils.widgets.text("table_name", "")
table_name = dbutils.widgets.get("table_name")

bronze_path = f"abfss://bronze@{storage_account}.dfs.core.windows.net/delta/{table_name}"
silver_path = f"abfss://silver@{storage_account}.dfs.core.windows.net/{table_name}"



In [0]:
#Table Configuration
table_config = {
    "customers":{
        "primary_key": "customer_id",
        "string_columns": ["customer_city", "customer_state"],
        "cast_columns":{
            "customer_zip_code_prefix": "integer"
        }
    },
    "orders" : {
        "primary_key": "order_id",
        "string_columns": ["order_status"],
        "cast_columns":{
            "order_purchase_timestamp": "timestamp",
            "order_approved_at": "timestamp",
            "order_delivered_carrier_date": "timestamp",
            "order_delivered_customer_date": "timestamp",
            "order_estimated_delivery_date": "timestamp"
        }
    },
    "payments" :{
        "primary_key":"order_id",
        "string_columns":["payment_type"],
        "cast_columns":{
            "payment_sequential":"integer",
            "payment_installments":"integer",
            "payment_value":"double"
        }
    },
    "order_items" : {
        "primary_key" : "order_id",
        "string_columns":[],
        "cast_columns":{
            "shipping_limit_date":"timestamp",
            "price":"double",
            "freight_value":"double"
        }
    },
    "products" : {
        "primary_key":"product_id",
        "string_columns":["product_category_name"],
        "cast_columns":{
            "product_name_length":"integer",
            "product_description_length":"integer",
            "product_photos_qty":"integer",
            "product_weight_g":"double",
            "product_length_cm":"double",
            "product_height_cm":"double",
            "product_width_cm":"double"
        },
        "rename_columns":{
            "product_name_lenght" : "product_name_length",
            "product_description_lenght" : "product_description_length"
        }
    }

}

  File <command-6948353973027171>, line 3
    "customers":{
                ^
SyntaxError: invalid syntax. Perhaps you forgot a comma?


In [0]:
#Helper Functions

#String Column Cleaning
def clean_string_columns(df, string_columns):
    for column in string_columns:
        df = df.withColumn(column, lower(trim(col(column))))
    return df

# Type casting
def cast_columns(df, cast_columns):
    for column, dtype in cast_columns.items():
        df = df.withColumn(column, col(column).cast(dtype))
    return df

#Rename Columns
def rename_columns(df, rename_cols):
    for old_col_name, new_col_name in rename_cols.items():
        df = df.withColumnRenamed(old_col_name, new_col_name)
    return df


In [0]:
#Data Quality Check Function
dq_results = {}

def run_dq_check(
    df,
    table_name,
    check_name,
    condition,
    severity="warning"
):

    failed_count = df.filter(condition).count()

    dq_results.setdefault(table_name, [])

    dq_results[table_name].append({
        "check_name": check_name,
        "severity": severity,
        "failed_count": failed_count
    })

    print(
        f"[{severity.upper()}] "
        f"{table_name} | "
        f"{check_name}: "
        f"{failed_count}"
    )

    if severity == "critical" and failed_count > 0:

        raise Exception(
            f"Critical DQ Failure | "
            f"Table: {table_name} | "
            f"Check: {check_name} | "
            f"Failed Count: {failed_count}"
        )

    return failed_count

In [0]:
# Duplicate Handling
def handle_duplicates(
    df,
    table_name,
    key_columns,
    severity="warning"
):

    before_count = df.count()

    deduped_df = df.dropDuplicates(key_columns)

    after_count = deduped_df.count()

    duplicates_removed = before_count - after_count

    dq_results.setdefault(table_name, [])

    dq_results[table_name].append({
        "check_name": f"duplicate_{'_'.join(key_columns)}",
        "severity": severity,
        "failed_count": duplicates_removed
    })

    print(
        f"[{severity.upper()}] "
        f"{table_name} | "
        f"duplicates_removed: "
        f"{duplicates_removed}"
    )

    return deduped_df

In [0]:
# Read the Table
df = spark.read.format("delta").load(bronze_path)
display(df)
df.printSchema()

In [0]:
config = table_config[table_name]

rename_cols = config.get("rename_columns", {})
df = rename_columns(df, rename_cols)
df = clean_string_columns(df, config["string_columns"])
df = cast_columns(df, config["cast_columns"])


In [0]:
#Table Specific Data Quality Checks

if table_name == "customers":
    run_dq_check(
        df, 
        table_name,                                 #Null Count
        f"null_count_{config['primary_key']}", 
        col(config['primary_key']).isNull(), 
        "critical"
        )
    
    df = handle_duplicates(df, table_name, [config["primary_key"]])    # Duplicates

    run_dq_check(
    df,
    table_name,
    "null_customer_city",
    col("customer_city").isNull(),
    "warning"
    )

    run_dq_check(
        df,
        table_name,
        "null_customer_state",
        col("customer_state").isNull(),
        "warning"
    )

In [0]:
if table_name == "orders":

    run_dq_check(
        df, 
        table_name,
        f"null_count_{config['primary_key']}",      #Null Count
        col(config['primary_key']).isNull(), 
        "critical"
        )
    
    df = handle_duplicates(df, table_name, [config["primary_key"]])    # Duplicates
    
    order_status_condition = ~col('order_status').isin(["shipped","canceled","approved","invoiced","delivered","unavailable","processing","created"])

    run_dq_check(
        df,
        table_name,
        "invalid_order_status_count", #Order_staus_Category
        order_status_condition
        )
    
    run_dq_check(
    df,
    table_name,
    "null_customer_id",
    col("customer_id").isNull(),
    "critical"
    )

    run_dq_check(
        df,
        table_name,
        "null_order_status",
        col("order_status").isNull(),
        "warning"
    )

In [0]:
if table_name == "payments":
    run_dq_check(
        df, 
        table_name,
        f"null_count_{config['primary_key']}",      #Null Count
        col(config['primary_key']).isNull(), 
        "critical"
        )
    
    df = handle_duplicates(df, table_name, ["order_id", "payment_sequential"])    # Duplicates
    
    run_dq_check(
        df,
        table_name,
        "invalid_payment_value_count", #Invalid Payment Value
        col("payment_value") < 0
        )
    
    run_dq_check(
        df,
        table_name,
        "invalid_payment_installments_count", #Invalid Payment Installments
        col("payment_installments") < 0
        )
    
    run_dq_check(
    df,
    table_name,
    "null_payment_sequential",
    col("payment_sequential").isNull(),
    "warning"
    )

    run_dq_check(
        df,
        table_name,
        "null_payment_type",
        col("payment_type").isNull(),
        "warning"
    )


In [0]:
if table_name == "order_items":
    run_dq_check(
        df, 
        table_name,
        "null_order_id",
        col("order_id").isNull(),
        "critical"
    )

    df = handle_duplicates(df, table_name, ["order_id", "order_item_id"])

    run_dq_check(
        df,
        table_name,
        "null_order_item_id",
        col("order_item_id").isNull(),
        "critical"
    )

    run_dq_check(
        df,
        table_name,
        "null_product_id",
        col("product_id").isNull(),
        "critical"
    )

    run_dq_check(
        df,
        table_name,
        "null_seller_id",
        col("seller_id").isNull(),
        "warning"
    )

    run_dq_check(
        df,
        table_name,
        "invalid_price_count", #Invalid Price
        col("price") < 0
        )
    
    run_dq_check(
        df,
        table_name,
        "invalid_freight_value_count", #Invalid Freight Value
        col("freight_value") < 0
        )

In [0]:
if table_name == "products":
    run_dq_check(
        df,
        table_name,
        "null_product_id",
        col("product_id").isNull(),
        "critical"
    )

    run_dq_check(
        df,
        table_name,
        "null_product_category",
        col("product_category_name").isNull(),
        "warning"
    )

    run_dq_check(
        df,
        table_name,
        "null_product_name_length",
        col("product_name_length").isNull(),
        "warning"
    )

    run_dq_check(
        df,
        table_name,
        "null_product_description_length",
        col("product_description_length").isNull(),
        "warning"
    )

    run_dq_check(
        df,
        table_name,
        "null_product_photos_qty",
        col("product_photos_qty").isNull(),
        "warning"
    )

    df = handle_duplicates(
        df,
        table_name,
        ["product_id"]
    )

    run_dq_check(
        df,
        table_name,
        "invalid_product_weight_g_count", #Invalid Product Weight
        col("product_weight_g") < 0
        )
    
    run_dq_check(
        df,
        table_name,
        "invalid_product_length_cm_count", #Invalid Product Length
        col("product_length_cm") < 0
        )
    
    run_dq_check(
        df,
        table_name,
        "invalid_product_height_cm_count", #Invalid Product Height
        col("product_height_cm") < 0
        )
    
    run_dq_check(
        df,
        table_name,
        "invalid_product_width_cm_count", #Invalid Product Width
        col("product_width_cm") < 0
        )
    
    run_dq_check(
        df,
        table_name,
        "invalid_product_weight_g_count", #Invalid Product Weight
        col("product_weight_g") < 0
        )
    
    run_dq_check(
        df,
        table_name,
        "invalid_product_length_cm_count", #Invalid Product Length
        col("product_length_cm") < 0
        )
    
    run_dq_check(
        df,
        table_name,
        "invalid_product_height_cm_count", #Invalid Product Height
        col("product_height_cm") < 0
        )
    
    run_dq_check(
        df,
        table_name,
        "invalid_product_width_cm_count", #Invalid Product Width
        col("product_width_cm") < 0
        )

In [0]:
# Write to silver Path
df.write.format("delta").mode("overwrite").save(silver_path)

In [0]:
dq_rows = []

for table_name, checks in dq_results.items():

    for check in checks:

        dq_rows.append(
            (
                datetime.now(),
                table_name,
                check["check_name"],
                check["severity"],
                check["failed_count"]
            )
        )

In [0]:
dq_schema = StructType([
    StructField("run_timestamp", TimestampType(), True),
    StructField("table_name", StringType(), True),
    StructField("check_name", StringType(), True),
    StructField("severity", StringType(), True),
    StructField("failed_count", IntegerType(), True)
])


In [0]:
dq_df = spark.createDataFrame(
    dq_rows,
    schema=dq_schema
)

In [0]:
dq_path = f"abfss://audit@{storage_account}.dfs.core.windows.net/silver_dq_logs/{table_name}"

In [0]:
dq_df.write.format("delta").mode("append").save(dq_path)

In [0]:
silver_df = spark.read.format("delta").load(silver_path)

print(f"Silver Row Count: {silver_df.count()}")

display(silver_df.limit(10))